# N2-02 Core experiment (identity + per-device attack detection + routing)
Runs PREREG_N2.md once. The notebook refuses to run if the embedded pre-registration differs from the signed hash.
Data: UNSW attack dataset (flowdata.zip, annotations.zip). Internet must be ON (downloads ~7 MB). No GPU needed.
Run all cells once. Do not edit the code between runs; changes after results are post hoc amendments.

In [ ]:
import json, hashlib, warnings
warnings.filterwarnings('ignore')
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
PREREG_TEXT = "# N2 pre-registration (FINAL, to be signed): identity, attack detection and identity-aware routing on the UNSW attack dataset\n\nExperiment N2. Separate from the 00d / CIC IoT-DIAD run. No 00d number, threshold, key or chain is reused.\nWritten: 2026-11-08. Written after the structure audits (01, 01b) and before any model was fitted or any performance number was seen.\n\n## 0. Declared history (read first)\n- 01b audit (structure only) judged this dataset against criteria written for a SUPERVISED attack-classifier design (>=30 attack instances on >=8 devices). It failed that rule.\n- After a literature check (Hamza thesis, arXiv 2001.10632; GeMID, arXiv 2411.14441) the design below was chosen instead: BENIGN-TRAINED per-device detectors, which do not need many attack examples. This is a different experiment, written after seeing the audit counts but before any performance. It is not a rescue of the failed rule and is recorded as such.\n- Attack-instance counts seen before writing (annotation rows + 1): Samsung camera 42, WeMo motion 36, Hue 30, Chromecast 27, WeMo switch 24, Netatmo camera 21, LIFX 15, Echo 9, iHome 3; TP-Link plug unread.\n\n## 1. Questions and hypotheses\nQ1 Identity: can a device be told apart from its network behaviour well enough to verify a claimed identity, reject other known devices and reject unknown devices?\nQ2 Attack detection: can a per-device detector trained only on benign traffic detect scripted attacks?\nQ3 Routing: does using the verified identity to pick a per-device detector beat the best global detector?\nQ4 Mismatch: what happens when the claimed device is not the actual device (states in section 8)?\nQ5 ML-security: how do poisoning, evasion/mimicry, backdoor and extraction affect these models (section 10)?\nNo hypothesis is guaranteed. Each is reported as pass or fail against its gate.\n\n## 2. Data\nUNSW IoT attack dataset (Hamza et al., SOSR 2019): flowdata.zip (per-device per-minute flow counters), annotations.zip (attack start/end), attackinfo.xlsx. Licence MIT-0 per the UNSW page.\nDevices (MAC suffix -> device): cohort 1 (2018-05-28 to 2018-07-18): ec1a59832811 WeMo motion, ec1a5979f489 WeMo switch, 00166cab6b88 Samsung camera, 50c7bf005639 TP-Link plug, 70ee50183443 Netatmo camera. Cohort 2 (2018-09-24 to 2018-10-26): 0017882b9a25 Hue, 44650d56ccd3 Echo, f4f5d88f0a3c Chromecast, 74c63b29d71d iHome, d073d5018308 LIFX.\nRow = one minute; Timestamp = epoch milliseconds, UTC.\nAnnotation files have no header; line = start_epoch_s, end_epoch_s, feature-tags, attack_name. Parse with rsplit so a missing/ragged file cannot be silently dropped; if any device's annotation file cannot be parsed, the run STOPS and reports it.\nAttack minute: Timestamp in [start*1000, end*1000] of any instance of that device. Attack name = family + rate (1/10/100 packets per second) + path tag (e.g. ArpSpoof100L2D).\nAttack neighbourhood: +/- 2 hours around each instance. Neighbourhood minutes are never used in TRAIN, CALIB or clean TEST.\n\n## 3. Features (identical rule for every model)\nCommon features: from each column name, direction (From/To) + location (Local/Internet) + protocol class (Tcp, Udp, Icmp, Arp, Other) + unit (Packet, Byte); sum all columns with the same key. 4 x 5 x 2 = 40 features, transformed with log1p. Unrecognised protocol tokens go to Other and are counted and reported.\nDetection input at minute m: the 40 features at m plus the trailing 5-minute mean of each (80 values). Trailing only: no future minutes.\nIdentity input: non-overlapping 15-minute windows (>=10 valid minutes) with mean, std and max of the 40 features (120 values). Windows touching an attack neighbourhood are excluded.\nForbidden inputs: absolute time, date, IP addresses, MAC addresses, column order, number of columns.\nNative-feature variant (device-specific columns): exploratory only, outside every gate.\n\n## 4. Splits (per device, by time)\nClean days = days with >=600 minute rows and not inside any attack neighbourhood +/- 1 day. In order: first 70% TRAIN, next 15% CALIB, last 15% clean-TEST, with a one-day gap between blocks. ATTACK-TEST = the minutes of each annotated instance (the detector input is built from the trailing window, which may include the 5 minutes before the instance).\nUnknown devices for identity: seed 20261108 picks one device from each cohort (sorted by MAC) to hold out. Held-out devices are absent from identity TRAIN and CALIB. Detection and routing use all 10 devices; identity verification applies to the 8 known ones, and a claim for a held-out device falls back to the global detector.\nDevices with fewer than 5 attack instances (iHome expected) are reported but excluded from the detection and routing gates.\n\n## 5. Identity module\nArms: RandomForest(500 trees) and XGBoost(500 trees, depth 6, lr 0.05), each with single-window decision or vote over 3 consecutive windows. Primary arm = best macro-F1 on CALIB among the four; chosen before clean-TEST is opened. Claim verification: accept a claim for device A if P(A) >= threshold_A, where threshold_A = 10th percentile of P(A) over device A's CALIB windows (so honest acceptance is about 0.90 on CALIB).\nGates on clean-TEST, primary arm only:\nG1 known-device macro-F1 >= 0.80\nG2 honest acceptance >= 0.80\nG3 random-mismatch rejection >= 0.95 (claim A, windows of a random other known device)\nG4 hard-mismatch rejection >= 0.90 (claim A, windows of the device most often confused with A on CALIB)\nG5 unknown-device impersonation rejection >= 0.60 (held-out devices claiming each known device)\nG6 false verification (any impersonation accepted) <= 0.10\nVerdict: POSITIVE if all pass; PARTIAL if G1 passes and at most one of G2-G6 fails; NEGATIVE otherwise.\nAlso reported: same arm on a leaky random 5-fold window split (to show inflation); confusion between same-vendor devices (WeMo motion/switch); results for same-cohort and cross-cohort mismatch pairs separately; 95% bootstrap CIs over days (1000 resamples, seed 20261108).\n\n## 6. Attack-detection module\nDetectors (all IsolationForest, 300 trees, seeds 11/22/33, scores averaged):\nE  per-device expert trained on that device's TRAIN minutes; alarm threshold = 99th percentile of that device's CALIB scores.\nG0 one global model trained on pooled TRAIN minutes of all 10 devices; single threshold = 99th percentile of pooled CALIB scores.\nG1 global model with the claimed device as a one-hot input, per-device threshold from CALIB (the strongest global baseline).\nAn instance is detected if >=2 minutes inside it raise an alarm.\nGate A1 (expert): macro instance-detection rate >= 0.80 over eligible devices AND mean alarm rate on clean-TEST minutes <= 0.02.\nAlso reported: detection by attack family and by rate (1/10/100 pps); alarm rate per device; per-instance outcomes.\n\n## 7. Routing rule (kill rule)\nRouting is accepted only if all hold: (E minus max(G0, G1)) macro instance-detection >= +0.05; paired bootstrap over instances (2000 resamples) lower 95% bound > 0; E's clean-TEST alarm rate <= the better global's + 0.005. Otherwise the verdict is routing_rejected_use_global and the system keeps the global detector. Reported either way.\n\n## 8. Mismatch states (what the gateway does)\nMATCH: claim verified (identity gate) -> route to the claimed device's expert if routing accepted, else global.\nMISMATCH: verification failed and the best-matching known device has P >= 0.60 -> flag, global detector.\nUNKNOWN: best P < 0.35 -> flag, global detector.\nUNCERTAIN: otherwise -> flag, global detector.\nUNREGISTERED_CLAIM: claimed device has no model (held-out) -> global detector.\nINPUT_REJECTED: schema or range invalid -> no decision.\nReported on clean-TEST and ATTACK-TEST: fraction of each state for (a) honest traffic, (b) random mismatch, (c) hard mismatch, (d) unknown device; and detection rate when the claimed device's expert is (wrongly) applied to another device's attack traffic.\n\n## 9. Run-once and amendments\nEach gate set is evaluated once. No threshold, feature, arm or split changes after clean-TEST or ATTACK-TEST results are seen. A change after a result may only be a labelled post hoc amendment and cannot change a verdict. A negative is reported as negative.\n\n## 10. ML-security stage (separate notebook, same registration)\nTargets: the identity classifier and the per-device detectors, on derived copies only.\nPoisoning: contaminate benign TRAIN with 1/5/10% attack or mislabelled minutes (>=3 seeds 11/22/33); measure change in instance detection, alarm rate and identity gates.\nEvasion / mimicry: (a) report detection vs attack rate (1/10/100 pps) from the data; (b) feature-space mimicry on identity: shift an impersonator's windows toward the target's centroid with budgets 0.1/0.25/0.5 of the valid range, query budgets 1,000 and 10,000, schema-valid perturbations only (min <= q <= max style constraints kept).\nBackdoor: trigger on a fixed subset of features in 1/5/10% of training windows, target = false MATCH for a chosen device; report attack success rate with a clean-control baseline.\nExtraction: surrogates (logistic regression, random forest, XGBoost) from 100 to 10,000 queries, labels / rounded / full probabilities.\nRules: defences not fitted on test data; no extraction-prevention claim if only query cost rose; every result reported including null results.\nGateway: signed bundle for the N2 models (own key, own chain) and a replay demo of clean-TEST and ATTACK-TEST minutes; startup refuses a tampered bundle.\n\n## 11. Known weaknesses stated in advance\n- 10 devices only, two recording periods, so each device has same-period neighbours only; cohort differences may help identity.\n- Scripted attacks at fixed rates in the same lab; detection may be easy and routing may have nothing to improve.\n- 3 to 42 attack instances per device; wide intervals expected.\n- Benign-trained detectors do not classify attack types; attack-type prediction is not claimed.\n- Unit-level identity of same-model devices cannot be tested here (all 10 are different models).\n- Training and attack days are not time-ordered for every device (attacks fall early in cohort 1); mild drift is possible.\n- Results are one dataset in one lab and do not establish general IoT security.\n\n## 12. Freeze\nSign the SHA-256 of this file with a NEW Ed25519 key (own chain, own _private folder) before any modelling notebook is run. The modelling notebook embeds this text and refuses to run if its hash differs from the signed one.\n"
CHAIN_BLOCK = "{\"hash\": \"e7b5f3bb8a47ac22f93c7530b590896eab695ea881994c3c0d4d46461504418d\", \"header\": {\"index\": 0, \"merkle_root\": \"ab23f4617892cc47814f869095a91b4b837679a7e3e1b7c4b4df49fc5206f5e7\", \"prev_hash\": \"0000000000000000000000000000000000000000000000000000000000000000\", \"signer_pub\": \"93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23\", \"timestamp\": 1794145734.8690827}, \"payload\": {\"experiment\": \"N2\", \"file\": \"PREREG_N2.md\", \"kind\": \"prereg_freeze\", \"note\": \"signed before any modelling; written after 01/01b structure audits\", \"sha256\": \"755fc38bbc786b5ad585e1a40ece04edb87adb9eac3cbd3514e750e9047126ad\", \"supersedes\": {\"PREREG_DRAFT_SUPERSEDED_identity_only.md\": \"5e05e5a7122ce7cc78c2840f75e40d12448cb7362bea4d1cf270ae5b0467c4a1\"}}, \"sig\": \"d436c49ed1307bef8152bae05b040bbc2948db2961e6a0cbcd12fab7f4910aa15792091d6905c25a7c8136d68714c506ef471900ce43ab2a02ac26674a804900\"}"
PUB = '93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23'
def _canon(o): return json.dumps(o, sort_keys=True, separators=(',', ':')).encode()
blk = json.loads(CHAIN_BLOCK)
sha = hashlib.sha256(PREREG_TEXT.encode()).hexdigest()
assert blk['payload']['sha256'] == sha, 'PREREG text differs from the signed version: STOP'
assert blk['header']['signer_pub'] == PUB
assert hashlib.sha256(_canon({'header': blk['header'], 'payload': blk['payload']})).hexdigest() == blk['hash'], 'block hash mismatch'
Ed25519PublicKey.from_public_bytes(bytes.fromhex(PUB)).verify(bytes.fromhex(blk['sig']), blk['hash'].encode())
print('PREREG signature OK | sha256', sha, '| chain head', blk['hash'])

## Library (embedded; same file that passed the synthetic smoke test)

In [ ]:
"""N2 core: identity + per-device attack detection + routing on the UNSW attack dataset.
Implements PREREG_N2.md exactly. Interpretations the prereg left open are marked INTERPRETATION."""
import os, re, json, glob, hashlib
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

DEVICES = {'ec1a59832811': ('WeMoMotion', 1), 'ec1a5979f489': ('WeMoSwitch', 1), '00166cab6b88': ('SamsungCamera', 1),
           '50c7bf005639': ('TPLinkPlug', 1), '70ee50183443': ('NetatmoCamera', 1),
           '0017882b9a25': ('HueBulb', 2), '44650d56ccd3': ('Echo', 2), 'f4f5d88f0a3c': ('Chromecast', 2),
           '74c63b29d71d': ('iHome', 2), 'd073d5018308': ('LIFX', 2)}
CFG = dict(split_seed=20261108, seeds=(11, 22, 33), if_trees=300, rf_trees=500, xgb_trees=500,
           boot=1000, boot_inst=2000, min_attack_inst=5)
COL_RE = re.compile(r'^(From|To)(Local|Internet)(.+?)(Packet|Byte)(?:\.\d+)?$')
FEATS = [f'{d}{l}_{p}_{u}' for d in ('From', 'To') for l in ('Local', 'Internet')
         for p in ('Tcp', 'Udp', 'Icmp', 'Arp', 'Other') for u in ('Packet', 'Byte')]

def proto_class(mid):
    for k in ('Tcp', 'Udp', 'Icmp', 'Arp'):
        if mid.startswith(k): return k
    return 'Other'

def common_features(df):
    groups, unmatched = {}, []
    for c in df.columns:
        if c == 'Timestamp': continue
        m = COL_RE.match(str(c))
        if not m: unmatched.append(c); continue
        d, l, mid, u = m.groups()
        groups.setdefault(f'{d}{l}_{proto_class(mid)}_{u}', []).append(c)
    X = pd.DataFrame(0.0, index=df.index, columns=FEATS)
    for k, cl in groups.items():
        X[k] = df[cl].apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0).sum(axis=1)
    other_cols = sum(len(v) for k, v in groups.items() if '_Other_' in k)
    return np.log1p(X), unmatched, other_cols

def minute_table(df):
    df = df.sort_values('Timestamp').reset_index(drop=True)
    X, unm, oc = common_features(df)
    M = X.rolling(5, min_periods=1).mean()
    Xn = X.to_numpy(np.float32)
    return dict(ts=df['Timestamp'].astype('int64').to_numpy(), X=Xn,
                F=np.hstack([Xn, M.to_numpy(np.float32)]), unmatched=unm, other_cols=oc)

ATK_RE = re.compile(r'^(.*?)(\d+)([LWD]2[LWD])$')
def parse_annotations(path):
    inst = []
    for line in open(path, errors='replace'):
        line = line.strip()
        if not line: continue
        h = line.split(',', 2)
        if len(h) < 3: raise ValueError(f'unparseable annotation line in {path}: {line[:80]}')
        s, e = float(h[0]), float(h[1])            # raises if a header sneaks in: STOP rule
        name = h[2].rpartition(',')[2].strip()
        inst.append((s, e, name))
    inst = sorted(set(inst))
    if not inst: raise ValueError(f'no instances in {path}')
    return inst

def attack_parts(name):
    m = ATK_RE.match(name)
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (name, -1, '')

def split_days(ts, inst, min_rows=600):
    day = (ts // 86400000).astype(int)
    neigh = np.zeros(len(ts), bool)
    for s, e, _ in inst:
        neigh |= (ts >= (s - 7200) * 1000) & (ts <= (e + 7200) * 1000)
    ad = set(day[neigh].tolist()); adb = ad | {d - 1 for d in ad} | {d + 1 for d in ad}
    u, c = np.unique(day, return_counts=True)
    clean = [int(d) for d, k in zip(u, c) if k >= min_rows and int(d) not in adb]
    n = len(clean)
    if n < 10: raise ValueError(f'only {n} clean days')
    a = int(0.70 * n); b = a + 1 + int(0.15 * n)
    return day, neigh, dict(train=set(clean[:a]), calib=set(clean[a + 1:b]), test=set(clean[b + 1:])), n

def load_all(flow_dir, ann_dir):
    D = {}
    for mac, (name, coh) in DEVICES.items():
        fp = glob.glob(f'{flow_dir}/**/{mac}_flowstats.csv', recursive=True)
        ap = glob.glob(f'{ann_dir}/**/{mac}.csv', recursive=True)
        if not fp or not ap: raise FileNotFoundError(f'{mac}: flow={bool(fp)} annotation={bool(ap)}')
        mt = minute_table(pd.read_csv(fp[0], low_memory=False))
        inst = parse_annotations(ap[0])
        day, neigh, sp, n = split_days(mt['ts'], inst)
        d = dict(name=name, cohort=coh, inst=inst, day=day, neigh=neigh, split=sp, n_clean=n, **mt)
        for k in ('train', 'calib', 'test'):
            d['m_' + k] = np.isin(day, list(sp[k])) & ~neigh
        D[mac] = d
    return D

# ---------------- detection ----------------
def fit_if(X, seeds, trees):
    return [IsolationForest(n_estimators=trees, random_state=s, n_jobs=-1).fit(X) for s in seeds]
def score_if(models, X):
    return np.mean([-m.score_samples(X) for m in models], axis=0)
def inst_hits(ts, sc, thr, inst, min_alarms=2):
    return np.array([int(((sc[(ts >= s * 1000) & (ts <= e * 1000)]) >= thr).sum() >= min_alarms) for s, e, _ in inst])

def run_detection(D, cfg):
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}
    oh = lambda m, n: np.tile(np.eye(len(macs), dtype=np.float32)[idx[m]], (n, 1))
    E = {m: fit_if(D[m]['F'][D[m]['m_train']], cfg['seeds'], cfg['if_trees']) for m in macs}
    G0 = fit_if(np.vstack([D[m]['F'][D[m]['m_train']] for m in macs]), cfg['seeds'], cfg['if_trees'])
    G1 = fit_if(np.vstack([np.hstack([D[m]['F'][D[m]['m_train']], oh(m, int(D[m]['m_train'].sum()))]) for m in macs]),
                cfg['seeds'], cfg['if_trees'])
    sE, s0, s1 = {}, {}, {}
    for m in macs:
        F = D[m]['F']; n = len(F)
        sE[m] = score_if(E[m], F); s0[m] = score_if(G0, F); s1[m] = score_if(G1, np.hstack([F, oh(m, n)]))
    thrE = {m: np.quantile(sE[m][D[m]['m_calib']], 0.99) for m in macs}
    thr0 = np.quantile(np.concatenate([s0[m][D[m]['m_calib']] for m in macs]), 0.99)
    thr1 = {m: np.quantile(s1[m][D[m]['m_calib']], 0.99) for m in macs}
    rows, per_inst = [], []
    for m in macs:
        d = D[m]; ts = d['ts']; te = d['m_test']
        hE = inst_hits(ts, sE[m], thrE[m], d['inst']); h0 = inst_hits(ts, s0[m], thr0, d['inst']); h1 = inst_hits(ts, s1[m], thr1[m], d['inst'])
        rows.append(dict(mac=m, device=d['name'], cohort=d['cohort'], n_inst=len(d['inst']), eligible=len(d['inst']) >= cfg['min_attack_inst'],
                         det_E=hE.mean(), det_G0=h0.mean(), det_G1=h1.mean(),
                         alarm_E=float((sE[m][te] >= thrE[m]).mean()), alarm_G0=float((s0[m][te] >= thr0).mean()), alarm_G1=float((s1[m][te] >= thr1[m]).mean())))
        for k, (s, e, nm) in enumerate(d['inst']):
            f, r, p = attack_parts(nm)
            per_inst.append(dict(mac=m, device=d['name'], name=nm, family=f, rate=r, path=p, hit_E=int(hE[k]), hit_G0=int(h0[k]), hit_G1=int(h1[k]),
                                 n_rows=int(((ts >= s * 1000) & (ts <= e * 1000)).sum())))
    det = pd.DataFrame(rows); pi = pd.DataFrame(per_inst)
    # wrong-expert: expert of A applied to device B's attack traffic
    wrong = []
    for B in macs:
        dB = D[B]
        for A in macs:
            if A == B: continue
            sc = score_if(E[A], dB['F'])
            wrong.append(dict(claimed=D[A]['name'], actual=dB['name'], same_cohort=D[A]['cohort'] == dB['cohort'],
                              det=float(inst_hits(dB['ts'], sc, thrE[A], dB['inst']).mean()),
                              eligible=len(dB['inst']) >= cfg['min_attack_inst']))
    return det, pi, pd.DataFrame(wrong)

def macro(det, col, elig=True):
    t = det[det.eligible] if elig else det
    return float(t[col].mean())

def routing_verdict(det, pi, cfg, seed=20261108):
    el = det[det.eligible]
    mE, m0, m1 = macro(det, 'det_E'), macro(det, 'det_G0'), macro(det, 'det_G1')
    best = 'G0' if m0 >= m1 else 'G1'; mG = max(m0, m1)
    aE = macro(det, 'alarm_E'); aG = macro(det, 'alarm_' + best)
    rng = np.random.default_rng(seed); macs = el.mac.tolist()
    hits = {m: pi[pi.mac == m][['hit_E', 'hit_' + best]].to_numpy() for m in macs}
    diffs = []
    for _ in range(cfg['boot_inst']):
        dd = []
        for m in macs:
            h = hits[m]; i = rng.integers(0, len(h), len(h)); dd.append(h[i, 0].mean() - h[i, 1].mean())
        diffs.append(np.mean(dd))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    accepted = bool((mE - mG) >= 0.05 and lo > 0 and aE <= aG + 0.005)
    gate_A1 = bool(mE >= 0.80 and aE <= 0.02)
    return dict(det_E=mE, det_G0=m0, det_G1=m1, best_global=best, delta=mE - mG, delta_ci95=[float(lo), float(hi)],
                alarm_E=aE, alarm_best_global=aG, gate_A1_expert=gate_A1,
                routing='routing_accepted' if accepted else 'routing_rejected_use_global', eligible_devices=el.device.tolist())

# ---------------- identity ----------------
def windows(d, days, win_ms=15 * 60000, min_rows=10):
    sel = np.isin(d['day'], list(days)) & ~d['neigh']
    if not sel.any(): return np.zeros((0, 120), np.float32), np.zeros(0, int)
    t = d['ts'][sel]; x = d['X'][sel]; df = pd.DataFrame(x); df['w'] = t // win_ms; df['day'] = d['day'][sel]
    g = df.groupby('w'); keep = g.size(); keep = keep[keep >= min_rows].index
    cols = list(range(x.shape[1]))
    agg = g[cols].agg(['mean', 'std', 'max']).loc[keep]
    F = agg.to_numpy(np.float32); F = np.nan_to_num(F)
    return F, g['day'].first().loc[keep].to_numpy()

def vote(P, k=3):
    if len(P) == 0: return P
    return pd.DataFrame(P).rolling(k, min_periods=1).mean().to_numpy()

def make_model(kind, cfg, n_classes):
    if kind == 'RF':
        return RandomForestClassifier(n_estimators=cfg['rf_trees'], class_weight='balanced', n_jobs=-1, random_state=11)
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=cfg['xgb_trees'], max_depth=6, learning_rate=0.05, objective='multi:softprob',
                         tree_method='hist', n_jobs=-1, random_state=11, num_class=n_classes)

def pick_unknown(D, seed):
    rng = np.random.default_rng(seed); out = []
    for coh in (1, 2):
        ms = sorted(m for m in D if D[m]['cohort'] == coh); out.append(ms[int(rng.integers(0, len(ms)))])
    return out

def cluster_boot(keys, vals, B, seed):
    keys = np.asarray(keys); vals = np.asarray(vals, float)
    u, inv = np.unique(keys, return_inverse=True)
    s = np.bincount(inv, vals); c = np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        i = rng.integers(0, len(u), len(u)); out.append(s[i].sum() / c[i].sum())
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

def state_of(P_row, claim, thr, known_idx):
    if claim is None: return 'UNREGISTERED_CLAIM'
    if P_row[claim] >= thr[claim]: return 'MATCH'
    pm = P_row.max()
    return 'MISMATCH' if pm >= 0.60 else ('UNKNOWN' if pm < 0.35 else 'UNCERTAIN')

def run_identity(D, cfg):
    unk = pick_unknown(D, cfg['split_seed']); known = sorted(m for m in D if m not in unk)
    kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    W = {}
    for m in D:
        for part in ('train', 'calib', 'test'):
            W[(m, part)] = windows(D[m], D[m]['split'][part])
            if len(W[(m, part)][0]) == 0:   # STOP rule: report, do not silently drop a device
                raise ValueError(f'no identity windows for {D[m]["name"]} in {part}')
    def stack(part, devs):
        Xs, ys, ds, dv = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); ds += list(dy); dv += [m] * len(F)
        return np.vstack(Xs), np.array(ys), np.array(ds), np.array(dv)
    Xtr, ytr, _, _ = stack('train', known); Xca, yca, dca, vca = stack('calib', known); Xte, yte, dte, vte = stack('test', known)
    Xun, _, dun, vun = stack('test', unk)
    arms = {}
    for kind in ('RF', 'XGB'):
        mdl = make_model(kind, cfg, K)
        if kind == 'XGB': mdl.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else: mdl.fit(Xtr, ytr)
        P = {n: mdl.predict_proba(X) for n, X in (('ca', Xca), ('te', Xte), ('un', Xun))}
        def per_dev_vote(P_, vv):
            out = P_.copy()
            for m in np.unique(vv):
                ii = np.where(vv == m)[0]; out[ii] = vote(P_[ii])
            return out
        for mode in ('single', 'vote3'):
            Q = P if mode == 'single' else {'ca': per_dev_vote(P['ca'], vca), 'te': per_dev_vote(P['te'], vte), 'un': per_dev_vote(P['un'], vun)}
            arms[f'{kind}-{mode}'] = dict(P=Q, f1_calib=float(f1_score(yca, Q['ca'].argmax(1), average='macro')))
    order = ['RF-single', 'RF-vote3', 'XGB-single', 'XGB-vote3']
    primary = max(order, key=lambda a: (round(arms[a]['f1_calib'], 12), -order.index(a)))
    Pa = arms[primary]['P']
    thr = np.array([np.quantile(Pa['ca'][yca == k, k], 0.10) for k in range(K)])
    pred = Pa['te'].argmax(1); res = dict(primary=primary, calib_macro_f1={a: arms[a]['f1_calib'] for a in order}, unknown_devices=unk, known_devices=known)
    G1 = float(f1_score(yte, pred, average='macro'))
    acc = (Pa['te'][np.arange(len(yte)), yte] >= thr[yte]).astype(float)
    per_dev_acc = {known[k]: float(acc[yte == k].mean()) for k in range(K)}
    G2 = float(np.mean(list(per_dev_acc.values())))              # INTERPRETATION: macro over known devices
    rng = np.random.default_rng(cfg['split_seed'])
    claim_r = np.array([rng.choice([k for k in range(K) if k != y]) for y in yte])
    rej_r = (Pa['te'][np.arange(len(yte)), claim_r] < thr[claim_r]).astype(float)
    hard = {}
    for A in range(K):
        mp = {B: Pa['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
    rows_h = [(i, A) for A in range(K) for i in np.where(yte == hard[A])[0]]
    rej_h = np.array([float(Pa['te'][i, A] < thr[A]) for i, A in rows_h]) if rows_h else np.array([1.0])
    key_h = [f'{vte[i]}_{dte[i]}' for i, A in rows_h]
    rej_u = np.array([[float(Pa['un'][i, A] < thr[A]) for A in range(K)] for i in range(len(Xun))]) if len(Xun) else np.zeros((1, K))
    rej_u_flat = rej_u.ravel(); key_u = np.repeat([f'{vun[i]}_{dun[i]}' for i in range(len(Xun))], K) if len(Xun) else ['x']
    G3 = float(rej_r.mean()); G4 = float(rej_h.mean()); G5 = float(rej_u_flat.mean())
    G6 = float(1 - np.concatenate([rej_r, rej_h, rej_u_flat]).mean())
    B = cfg['boot']; kt = [f'{vte[i]}_{dte[i]}' for i in range(len(yte))]
    ci = dict(G1=cluster_boot(kt, (pred == yte).astype(float), B, 1)[:], G2=cluster_boot(kt, acc, B, 2), G3=cluster_boot(kt, rej_r, B, 3),
              G4=cluster_boot(key_h, rej_h, B, 4), G5=cluster_boot(key_u, rej_u_flat, B, 5))
    ci['G1'] = 'accuracy CI shown (macro-F1 CI not bootstrapped)'; ci['note'] = 'cluster bootstrap over device-days'
    gates = dict(G1=G1 >= 0.80, G2=G2 >= 0.80, G3=G3 >= 0.95, G4=G4 >= 0.90, G5=G5 >= 0.60, G6=G6 <= 0.10)
    nf = sum(not v for k, v in gates.items() if k != 'G1')
    verdict = 'POSITIVE' if all(gates.values()) else ('PARTIAL' if gates['G1'] and nf <= 1 else 'NEGATIVE')
    # leaky comparison
    allX = np.vstack([Xtr, Xca, Xte]); ally = np.concatenate([ytr, yca, yte]); fl = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(allX, ally):
        mdl = make_model(primary.split('-')[0], cfg, K)
        (mdl.fit(allX[a], ally[a], sample_weight=compute_sample_weight('balanced', ally[a])) if primary.startswith('XGB') else mdl.fit(allX[a], ally[a]))
        fl.append(f1_score(ally[b], mdl.predict(allX[b]), average='macro'))
    # states
    def states(P_rows, claims):
        return pd.Series([state_of(P_rows[i], claims[i], thr, kidx) for i in range(len(P_rows))]).value_counts(normalize=True).round(4).to_dict()
    S = dict(honest=states(Pa['te'], list(yte)), random_mismatch=states(Pa['te'], list(claim_r)),
             hard_mismatch=states(Pa['te'][[i for i, A in rows_h]], [A for i, A in rows_h]) if rows_h else {},
             unknown_device=states(np.repeat(Pa['un'], K, axis=0), list(np.tile(np.arange(K), len(Xun)))) if len(Xun) else {})
    confusion = pd.crosstab(pd.Series([known[i] for i in yte], name='actual'), pd.Series([known[i] for i in pred], name='pred'))
    res.update(G1_macro_f1=G1, G2_honest_acceptance_macro=G2, G2_micro=float(acc.mean()), G3_random_mismatch_rejection=G3,
               G4_hard_mismatch_rejection=G4, G5_unknown_rejection=G5, G6_false_verification=G6, gates=gates, verdict=verdict,
               per_device_acceptance=per_dev_acc, hard_partner={known[A]: known[hard[A]] for A in range(K)}, ci95=ci,
               leaky_random_5fold_macro_f1=float(np.mean(fl)), states=S, n_windows=dict(train=len(Xtr), calib=len(Xca), test=len(Xte), unknown_test=len(Xun)))
    return res, confusion

## Download data

In [ ]:
import os, zipfile, shutil, requests, glob, hashlib
OUT = '/kaggle/working/n2_02_core'; os.makedirs(OUT, exist_ok=True)
DATA = '/kaggle/working/unsw_attack'; os.makedirs(DATA, exist_ok=True)
BASE = 'https://iotanalytics.unsw.edu.au/anomaly-data/'
hashes = {}
for n in ['flowdata.zip', 'annotations.zip']:
    p = (glob.glob(f'/kaggle/input/**/{n}', recursive=True) or [None])[0]
    if p is None:
        p = f'{DATA}/{n}'
        with requests.get(BASE + n, stream=True, timeout=600, headers={'User-Agent': 'ris-n2'}) as r:
            r.raise_for_status()
            with open(p, 'wb') as fh: shutil.copyfileobj(r.raw, fh)
    hashes[n] = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    with zipfile.ZipFile(p) as z: z.extractall(f'{DATA}/{n[:-4]}')
print(hashes)
json.dump(hashes, open(f'{OUT}/data_hashes.json', 'w'), indent=1)

## Load, split, detect, route

In [ ]:
D = load_all(f'{DATA}/flowdata', f'{DATA}/annotations')
info = pd.DataFrame([dict(device=d['name'], cohort=d['cohort'], rows=len(d['ts']), clean_days=d['n_clean'],
                          train_days=len(d['split']['train']), calib_days=len(d['split']['calib']), test_days=len(d['split']['test']),
                          instances=len(d['inst']), unmatched_cols=len(d['unmatched']), other_cols=d['other_cols']) for d in D.values()])
print(info.to_string(index=False)); info.to_csv(f'{OUT}/data_summary.csv', index=False)
det, pi, wrong = run_detection(D, CFG)
rv = routing_verdict(det, pi, CFG)
det.to_csv(f'{OUT}/detection_by_device.csv', index=False); pi.to_csv(f'{OUT}/detection_by_instance.csv', index=False); wrong.to_csv(f'{OUT}/wrong_expert_detection.csv', index=False)
print(det.round(3).to_string(index=False)); print(json.dumps(rv, indent=1, default=float))
print(pi.groupby('rate')[['hit_E', 'hit_G0', 'hit_G1']].mean().round(3))
print(pi.groupby('family')[['hit_E', 'hit_G0', 'hit_G1']].agg(['mean']).round(3))
print('instances with no rows:', int((pi.n_rows == 0).sum()))
print(wrong[wrong.eligible].groupby('same_cohort').det.mean())

## Identity, mismatch states

In [ ]:
ident, conf = run_identity(D, CFG)
conf.to_csv(f'{OUT}/identity_confusion.csv')
print(json.dumps(ident, indent=1, default=str))

## Save

In [ ]:
final = dict(prereg_sha256=sha, chain_head=blk['hash'], data_hashes=hashes, cfg={k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
             identity=ident, routing=rv, interpretations=['G2 honest acceptance = macro over known devices (micro also reported)',
             'G3 random mismatch: one random claimed device per window, seed split_seed', 'G6 pooled over random, hard and unknown impersonation attempts',
             'instances with no rows count as missed', 'macro-F1 CI not bootstrapped; accuracy CI shown'])
json.dump(final, open(f'{OUT}/results.json', 'w'), indent=1, default=str)
shutil.make_archive('/kaggle/working/n2_02_core_results', 'zip', OUT)
print('IDENTITY:', ident['verdict'], ident['gates'], '| ROUTING:', rv['routing'], '| EXPERT GATE A1:', rv['gate_A1_expert'])